In [2]:
import os
import torch 
import pandas as pd
import numpy as np

In [3]:
#config 
results_folder = '/home/rachel/Desktop/mm-vae results/multiregion_adbxd_cr_figureResults/multiregion_samecenters_seed10_trainosd50_V3'
dataset_abbrev = ['HC', 'PFC'] #['HC1', 'HC2'] # ['Geno', 'Physio']
epoch = 10000
latent_dim = 10 # 20 
hues = ['encoder', 'bin']
random_seed = 22
model_type = 'CGMVAE'

In [4]:
latent_space_1 = pd.read_csv(os.path.join(results_folder, f'latent_variables_epoch{epoch}_vae{dataset_abbrev[0]}.csv'), index_col=False)
latent_space_1

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,bin,cfm,encoder,strain,14-6-residual
0,-8.797236,-4.894272,2.111475,8.519982,12.791437,18.410599,25.216757,29.190458,37.120240,41.817627,3,1.243151,HC,124,1.277393
1,-16.458345,-11.771194,-7.174741,-3.625618,3.943302,6.394501,11.820374,16.571030,24.964413,30.362510,3,1.243151,HC,124,1.277393
2,-18.091574,-13.185998,-8.856009,-5.383374,2.315673,4.631015,9.943457,14.564608,23.036581,28.362236,3,1.243151,HC,124,1.277393
3,-9.204573,-5.152271,1.954065,8.411310,13.084518,18.526867,25.465850,29.619830,37.719894,42.480026,3,1.243151,HC,124,1.277393
4,-15.862563,-11.323180,-6.809146,-3.258895,4.118066,6.458795,11.666695,16.222832,24.655697,29.856333,3,1.243151,HC,124,1.277393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9746,-20.147890,-12.936576,-15.897243,-9.085006,-3.124617,-1.909373,2.309832,6.215364,9.963960,15.217135,1,-0.215123,HC,28,-0.576731
9747,-20.031136,-13.460803,-14.630583,-8.228660,-2.012851,-1.725156,3.041983,6.744488,10.722353,15.584701,1,-0.215123,HC,28,-0.576731
9748,-19.752373,-12.849339,-15.275242,-8.791967,-2.870516,-1.942967,2.364191,5.998524,10.046764,15.187144,1,-0.215123,HC,28,-0.576731
9749,-19.214336,-12.885562,-16.888690,-10.685880,-4.761407,-3.497963,0.719140,4.472291,8.508114,13.720981,1,-0.215123,HC,28,-0.576731


In [5]:
latent_space_2 = pd.read_csv(os.path.join(results_folder, f'latent_variables_epoch{epoch}_vae{dataset_abbrev[1]}.csv'), index_col=False)
latent_space_2

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,bin,cfm,encoder,strain,14-6-residual
0,-9.371390,-5.279217,1.825949,8.192675,12.983744,18.338358,25.234829,29.425016,37.419060,42.351322,3,1.243151,PFC,124,1.277393
1,-7.649703,-3.587802,3.502619,9.954540,14.706524,20.043278,26.936472,31.060795,39.173985,44.016450,3,1.243151,PFC,124,1.277393
2,-9.944201,-5.787368,1.307106,7.820934,12.642464,18.054169,25.006240,29.036854,37.465680,42.161232,3,1.243151,PFC,124,1.277393
3,-7.826996,-3.696502,3.423266,9.914746,14.739337,20.124537,27.051073,31.093939,39.471485,44.185020,3,1.243151,PFC,124,1.277393
4,-8.754727,-4.593140,2.531374,8.915482,13.790705,19.159216,26.099195,30.300007,38.365030,43.345444,3,1.243151,PFC,124,1.277393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9746,-23.748928,-16.342093,-17.412416,-11.260060,-5.209814,-4.162101,0.236154,3.957271,7.941266,13.136404,1,-0.215123,PFC,28,-0.576731
9747,-23.798943,-17.435293,-17.162943,-10.865115,-4.886643,-4.217895,0.473804,4.189680,8.058167,12.911073,1,-0.215123,PFC,28,-0.576731
9748,-26.468983,-18.748316,-18.569887,-12.086290,-5.969710,-4.602660,-0.120975,3.915201,7.974979,13.397053,1,-0.215123,PFC,28,-0.576731
9749,-24.699724,-18.313803,-15.697443,-9.139642,-3.504307,-2.564813,1.612955,5.109154,9.309296,14.297555,1,-0.215123,PFC,28,-0.576731


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = torch.load(os.path.join(results_folder, f'saved_model_epoch{epoch}.pth'), map_location=device, weights_only=False)

In [19]:
import torch
import numpy as np

def batched_decode_mse(vae_model, latent_df, actual_data, latent_dim, batch_size, device):
    """
    Decodes in mini-batches to match training BN dynamics and calculates MSE.
    """
    vae_model.train() # Keeping BN in training mode to check batch-dependency
    
    # Prepare data
    z_all = torch.tensor(latent_df.iloc[:, :latent_dim].values, dtype=torch.float32)
    bins_all = torch.tensor(latent_df['bin'].values, dtype=torch.long)
    
    num_samples = z_all.size(0)
    all_reconstructions = []

    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            # Extract batch
            z_batch = z_all[i:i+batch_size].to(device)
            bins_batch = bins_all[i:i+batch_size].to(device)
            
            # Decode specific batch
            recon_batch, _ = vae_model.decode(z_batch, bins_batch)
            
            # Append results (removing potential extra columns like 'bin' if appended in model)
            # Adjust indexing [:, :-1] based on your specific output structure
            all_reconstructions.append(recon_batch[:, -1].detach().cpu().numpy())

    # Concatenate all batches back together
    full_recon = np.concatenate(all_reconstructions, axis=0)
    print(full_recon)
    print(actual_data)
    
    # Calculate MSE against the actual ground truth
    mse = np.mean(np.square(full_recon - actual_data[:num_samples]))
    return mse, full_recon

In [8]:
norm = pd.read_csv('/home/rachel/Desktop/mm-vae data/multiregion_adbxd_cr/hc_pfc_adbxd_paired_30perc.csv')

/tmp/ipykernel_18019/940678472.py:1: DtypeWarning: Columns (9712) have mixed types. Specify dtype option on import or set low_memory=False.
  norm = pd.read_csv('/home/rachel/Desktop/mm-vae data/multiregion_adbxd_cr/hc_pfc_adbxd_paired_30perc.csv')


In [16]:
#actual_m1 = norm.iloc[:, :4842].values
actual_m1 = latent_space_1.iloc[:, -4]

In [17]:
# actual_m2 = norm.iloc[:, 4842+10:-11].values
actual_m2 = latent_space_2.iloc[:, -4]

In [20]:
# --- Example Usage ---
# Use the same batch size you used during the training loop
TRAIN_BATCH_SIZE = 271

mse_m1_ls1, _ = batched_decode_mse(vae.vaes[0], latent_space_1, actual_m1, latent_dim, TRAIN_BATCH_SIZE, device)
mse_m1_ls2, _ = batched_decode_mse(vae.vaes[0], latent_space_2, actual_m1, latent_dim, TRAIN_BATCH_SIZE, device)
mse_m2_ls2, _ = batched_decode_mse(vae.vaes[1], latent_space_2, actual_m2, latent_dim, TRAIN_BATCH_SIZE, device)
mse_m2_ls1, _ = batched_decode_mse(vae.vaes[1], latent_space_1, actual_m2, latent_dim, TRAIN_BATCH_SIZE, device)

print(f"Batched MSE M1-LS1: {mse_m1_ls1:.6f}")
print(f"Batched MSE M1-LS2: {mse_m1_ls2:.6f}")

[ 0.91189015 -2.2747056  -2.251902   ... -0.73109424  0.28714612
 -1.4844716 ]
0       1.243151
1       1.243151
2       1.243151
3       1.243151
4       1.243151
          ...   
9746   -0.215123
9747   -0.215123
9748   -0.215123
9749   -0.215123
9750   -0.215123
Name: cfm, Length: 9751, dtype: float64
[ 0.7098229  0.3092963  1.5540657 ...  1.0459459 -1.4037986  0.2752625]
0       1.243151
1       1.243151
2       1.243151
3       1.243151
4       1.243151
          ...   
9746   -0.215123
9747   -0.215123
9748   -0.215123
9749   -0.215123
9750   -0.215123
Name: cfm, Length: 9751, dtype: float64
[ 1.8642949   0.11077539 -0.06554264 ...  2.589966   -0.65880793
  1.3546351 ]
0       1.243151
1       1.243151
2       1.243151
3       1.243151
4       1.243151
          ...   
9746   -0.215123
9747   -0.215123
9748   -0.215123
9749   -0.215123
9750   -0.215123
Name: cfm, Length: 9751, dtype: float64
[-0.04112773 -0.16341545  0.5429367  ...  0.05806275  1.5486821
  1.4431865 ]
0       1.2